In [1]:
from bayesian_spread_model import basic_spread_model, off_def_model, prepare_data, fit, extract_team_ratings

/Users/msubbaiah/Documents/march_madness/2026/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pandas as pd
import numpy as np
from jax import numpy as jnp

In [3]:
df = pd.read_csv("data/ncaabb25.csv")
df = df.dropna(subset=["line"])
df["neutral"] = df["neutral"].fillna(0)
df['home'] = df['home'].str.lower()
df['road'] = df['road'].str.lower()

In [4]:
team_spells = pd.read_csv("data/MTeamSpellings.csv",encoding='cp1252')
team_conf = pd.read_csv("data/MTeamConferences.csv")

In [5]:
fj = team_spells.merge(team_conf, on="TeamID", how="left")
fj_26 = fj[fj["Season"] == 2026]

In [6]:
fj_26.head()

,TeamNameSpelling,TeamID,Season,ConfAbbrev
23,a&m-corpus chris,1394,2026,southland
47,a&m-corpus christi,1394,2026,southland
60,abilene chr,1101,2026,wac
73,abilene christian,1101,2026,wac
86,abilene-christian,1101,2026,wac


In [7]:
df_test = df.merge(fj_26[["TeamNameSpelling", "ConfAbbrev"]], 
                   left_on="home", 
                   right_on="TeamNameSpelling", 
                   how="left").drop(columns=["TeamNameSpelling"]).rename(columns={"ConfAbbrev": "home_conf"})
df_test = df_test.merge(fj_26[["TeamNameSpelling", "ConfAbbrev"]],
                     left_on="road", 
                     right_on="TeamNameSpelling", 
                     how="left").drop(columns=["TeamNameSpelling"]).rename(columns={"ConfAbbrev": "road_conf"})

In [8]:
df_test['home_conf'] = df_test['home_conf'].fillna("Unknown")
df_test['road_conf'] = df_test['road_conf'].fillna("Unknown")

In [9]:
model_data_dct, teams, confs, team_to_idx, conf_to_idx = prepare_data(df_test)

In [11]:
model_data_dct.pop('line')
#model_data_dct.pop('home_win')

Array([ 8.5, 35.5, -3.5, ...,  6. ,  3.5,  9.5], dtype=float32)

In [12]:
mod, mod_samples = fit(data = model_data_dct, model_func = off_def_model)

/Users/msubbaiah/Documents/march_madness/2026/bayesian_spread_model.py:106: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(kernel, num_warmup=num_warmup,
sample: 100%|██████████| 2000/2000 [00:14<00:00, 140.63it/s, 127 steps of size 8.15e-02. acc. prob=0.89]


In [13]:
def team_ratings(samples, teams):
    off  = samples["off"]    # (draws, T)
    deff = samples["def"]   # (draws, T)
    
    net = off + deff

    return pd.DataFrame({
        "team":      teams,
        "off_mean":  np.array(jnp.mean(off,  axis=0)),
        "off_p5":    np.array(jnp.percentile(off,  5,  axis=0)),
        "off_p95":   np.array(jnp.percentile(off,  95, axis=0)),
        "def_mean":  np.array(jnp.mean(deff, axis=0)),
        "def_p5":    np.array(jnp.percentile(deff, 5,  axis=0)),
        "def_p95":   np.array(jnp.percentile(deff, 95, axis=0)),
        "net_mean":  np.array(jnp.mean(net,  axis=0)),
        "net_p5":    np.array(jnp.percentile(net,  5,  axis=0)),
        "net_p95":   np.array(jnp.percentile(net,  95, axis=0)),
    }).sort_values("net_mean", ascending=False).reset_index(drop=True)

In [14]:
def conference_ratings(samples, confs):
    mu_off = samples["mu_off_conf"]   
    mu_def = samples["mu_def_conf"]   
    net    = mu_off + mu_def

    return pd.DataFrame({
        "conference": confs,
        "off_mean":   np.array(jnp.mean(mu_off, axis=0)),
        "off_p5":     np.array(jnp.percentile(mu_off, 5,  axis=0)),
        "off_p95":    np.array(jnp.percentile(mu_off, 95, axis=0)),
        "def_mean":   np.array(jnp.mean(mu_def, axis=0)),
        "def_p5":     np.array(jnp.percentile(mu_def, 5,  axis=0)),
        "def_p95":    np.array(jnp.percentile(mu_def, 95, axis=0)),
        "net_mean":   np.array(jnp.mean(net,    axis=0)),
        "net_p5":     np.array(jnp.percentile(net,    5,  axis=0)),
        "net_p95":    np.array(jnp.percentile(net,    95, axis=0)),
    }).sort_values("net_mean", ascending=False).reset_index(drop=True)

In [15]:
conf_rtgs = conference_ratings(mod_samples, confs)
conf_rtgs

,conference,off_mean,off_p5,off_p95,def_mean,def_p5,def_p95,net_mean,net_p5,net_p95
0,SEC,7.064541,3.875675,10.470235,1.025859,-1.249489,3.702814,8.090401,3.571846,13.250337
1,ACC,4.184791,1.280929,7.204386,1.950035,-0.327913,4.778135,6.134826,1.750443,11.065061
2,BIG_TEN,3.429003,0.497424,6.454890,2.675667,0.035201,5.599231,6.104671,1.382427,11.211185
3,BIG_TWELVE,3.731490,0.805813,6.879219,1.886071,-0.460194,4.855781,5.617561,1.186210,10.910734
4,BIG_EAST,1.411649,-1.833889,4.813967,1.286990,-1.235222,4.389451,2.698638,-2.125931,8.149679
5,MWC,0.859679,-2.147155,3.913415,0.240114,-2.127401,2.688134,1.099793,-3.233450,5.522502
6,WCC,0.342548,-2.904909,3.549831,0.685192,-1.715352,3.307600,1.027739,-3.639755,5.812974
7,HORIZON,1.512402,-1.664926,4.740906,-0.512680,-3.071871,1.809128,0.999721,-3.553464,5.378273
8,A_TEN,-0.725682,-3.752752,2.310738,1.570274,-0.774918,4.454096,0.844592,-3.827417,5.691601
9,MVC,0.395955,-2.932728,3.653198,0.448291,-1.966293,3.050304,0.844245,-3.963896,5.770036


In [16]:
mod_samples.keys()

dict_keys(['L_corr', 'alpha', 'def', 'mu_def_conf', 'mu_intercept', 'mu_off_conf', 'off', 'sigma', 'sigma_def', 'sigma_def_conf', 'sigma_off', 'sigma_off_conf', 'z'])

In [17]:
team_rtgs = team_ratings(mod_samples, teams)
# old model 
# team_rtgs = extract_team_ratings(mod_samples, model_data_dct["teams"])

In [18]:
team_rtgs

,team,off_mean,off_p5,off_p95,def_mean,def_p5,def_p95,net_mean,net_p5,net_p95
0,DUKE,8.203073,4.933310,11.593808,14.595005,11.362392,17.812628,22.798075,17.421165,28.395159
1,FLORIDA,13.188945,9.718700,16.822758,7.034450,3.605399,10.525833,20.223394,14.581679,26.138697
2,ARIZONA,11.248633,7.820737,14.854831,7.798825,4.467041,11.170627,19.047457,13.355369,25.071199
3,MICHIGAN,9.000638,5.686205,12.496022,9.164566,5.906331,12.531787,18.165203,12.713619,24.088520
4,IOWA ST.,5.539030,2.114012,9.039965,12.130810,8.900832,15.484946,17.669840,12.069441,23.551319
...,...,...,...,...,...,...,...,...,...,...
360,NORTHERN ILLINOIS,-9.996612,-13.696289,-6.400383,-4.945006,-8.436934,-1.541404,-14.941618,-20.668406,-9.303498
361,MISS VALLEY ST.,-9.455703,-12.886748,-5.994540,-6.397226,-9.813315,-3.099909,-15.852928,-21.570492,-10.228785
362,WESTERN ILLINOIS,-9.341971,-13.547706,-5.169274,-7.446028,-11.632774,-3.433650,-16.787998,-23.890652,-9.964464
363,AIR FORCE,-10.540997,-14.067345,-7.015890,-6.501748,-9.889256,-2.935921,-17.042746,-22.658251,-11.297843
